In [1]:
from functools import partial

import ee
import geemap
from pathlib import Path
import numpy as np
import geopandas as gpd
import multiprocessing

import TerrainChips
import LULCChips
import SlopeAndAspect
import SatEmbeddingChips
import GetS2Chips


from IPython.display import clear_output

In [2]:
# ee.Authenticate(force=True) # use this if you need to change ee account
_ = geemap.Map()

In [3]:
# targetOutputPath
dataPath = Path('.')

In [4]:
targetPath = dataPath / 'NodeFeatures'
targetDSMPath = targetPath / 'TerrainStats' / 'JAXA_ALOS_AW3D30_V3_2'
targetDEMPath = targetPath / 'TerrainStats' / 'COPERNICUS_DEM_GLO30'
targetAspectPath = targetPath / 'TerrainStats' / 'COPERNICUS_DEM_GLO30_Aspect'
targetSlopePath = targetPath / 'TerrainStats' / 'COPERNICUS_DEM_GLO30_Slope'

targetS2Path = targetPath/'Satellite'/'Seasonal'/'S2'


targetLCLUPath = targetPath/'LCLUStats'
LCLUTargetPath = targetLCLUPath/'ESA_WorldCover_10m_v200'

targetFoundationModelEmbeddingPath = targetPath/'FoundationModelEmbedding'
GoogleEmbeddingTargetPath = targetFoundationModelEmbeddingPath/'GOOGLE_SATELLITE_EMBEDDING_V1_ANNUAL'

targetDSMPath.mkdir(exist_ok=True, parents=True)
targetDEMPath.mkdir(exist_ok=True, parents=True)
targetAspectPath.mkdir(exist_ok=True, parents=True)
targetSlopePath.mkdir(exist_ok=True, parents=True)

LCLUTargetPath.mkdir(exist_ok=True, parents=True)
GoogleEmbeddingTargetPath.mkdir(exist_ok=True, parents=True)
targetS2Path.mkdir(exist_ok=True, parents=True)

In [5]:
# In my case I have a shapefile with all the points, but as long as you create coords it doesn't really matter
stations = gpd.read_file('path_to_shapefile')
stat_coords = list(stations['geometry'])
stat_lons = np.array([i.x for i in stat_coords]).astype(np.float32)
stat_lats = np.array([i.y for i in stat_coords]).astype(np.float32)

coords = list(zip(stat_lons, stat_lats))

In [6]:
base_chipSize = 32 # number of pixels at 30 meters resolution
chipSize = base_chipSize
geeterraindata = ee.ImageCollection("JAXA/ALOS/AW3D30/V3_2")
layer = 'DSM'

crs = 'EPSG:4326'
scale = 30

cpus = 20

with multiprocessing.Pool(cpus) as pool:
    pool.map(partial(TerrainChips.DownloadAndProcessChips, chipSize=chipSize, targetFolder=targetDSMPath, terrain=geeterraindata, layer=layer, crs=crs, scale=scale), coords)
    # clear_output(wait=True)

In [7]:
chipSize = base_chipSize

geeterraindata = ee.ImageCollection("COPERNICUS/DEM/GLO30")
layer = 'DEM'

crs = 'EPSG:4326'
scale = 30

cpus = 20

with multiprocessing.Pool(cpus) as pool:
    pool.map(partial(TerrainChips.DownloadAndProcessChips, chipSize=chipSize, targetFolder=targetDEMPath, terrain=geeterraindata, layer=layer, crs=crs, scale=scale), coords)
    # clear_output(wait=True)

In [8]:
chipSize = base_chipSize

geeterraindata = ee.ImageCollection("COPERNICUS/DEM/GLO30").select('DEM')

crs = 'EPSG:4326'
scale = 30

cpus = 20

with multiprocessing.Pool(cpus) as pool:
    pool.map(partial(SlopeAndAspect.DownloadAndProcessChips, chipSize=chipSize, targetFolder=targetSlopePath, terrain=geeterraindata, crs=crs, scale=scale, type='slope'), coords)

clear_output(wait=True)
with multiprocessing.Pool(cpus) as pool:
    pool.map(partial(SlopeAndAspect.DownloadAndProcessChips, chipSize=chipSize, targetFolder=targetAspectPath, terrain=geeterraindata, crs=crs, scale=scale, type='aspect'), coords)

In [9]:
chipSize = base_chipSize * 3 # I want the same extent for 10 and 30 meters resolution, so I use 3* the number of pixels at 30 meters

geelcludata = ee.ImageCollection("ESA/WorldCover/v200")

crs = 'EPSG:4326'
scale = 10
    
with multiprocessing.Pool(cpus) as pool:
    pool.map(partial(LULCChips.DownloadAndProcessChips, chipSize=chipSize, targetFolder=LCLUTargetPath, lclu=geelcludata, crs=crs, scale=scale), coords)

In [10]:
chipSize = 1

crs = 'EPSG:4326'
scale = 10
    
with multiprocessing.Pool(cpus) as pool:
    pool.map(partial(SatEmbeddingChips.DownloadAndProcessChips, chipSize=chipSize, targetFolder=GoogleEmbeddingTargetPath, crs=crs, scale=scale), coords)

In [11]:
chipSize = base_chipSize * 3
s2data = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
s2data = s2data.map(lambda image: image.reproject('EPSG:4326'))

crs = 'EPSG:4326'
scale = 10

cpus = 20

with multiprocessing.Pool(cpus) as pool:
    pool.map(partial(GetS2Chips.DownloadS2Chip, chipSize=chipSize, targetFolder=targetS2Path, crs=crs, scale=scale, year_start=2024, year_end=2024), coords)
    # clear_output(wait=True)—